# Entrainement U-Net (caries) sur GPU en ligne

A executer sur **Google Colab** (Runtime > Modifier le type d'execution > GPU).
Ce notebook clone le repo, telecharge le dataset DC1000, entraine le U-Net,
evalue face a la baseline, compare des hyperparametres, puis permet de
recuperer le modele entraine.

In [ ]:
# 1. Recuperer le code
!git clone https://github.com/khalil-ab/cariesDetectionAssistant.git
%cd cariesDetectionAssistant
!pip install -q torch torchvision opencv-python-headless scikit-learn mlflow matplotlib

In [ ]:
# 2. Telecharger le dataset DC1000 (source Google Drive) et le decompresser
!pip install -q gdown
!gdown 1Xn1oGHvhGF9GbkcLEtCOV5QvWWqt1y62 -O dc1000.zip
!unzip -q dc1000.zip -d data_dc1000
!ls data_dc1000

In [ ]:
# 3. Pointer la config vers les donnees telechargees
import os, glob
candidats = glob.glob('data_dc1000/**/train/images', recursive=True)
racine = os.path.dirname(os.path.dirname(candidats[0]))
os.environ['CARIES_DATA_ROOT'] = os.path.abspath(racine)
print('CARIES_DATA_ROOT =', os.environ['CARIES_DATA_ROOT'])

In [ ]:
# 4. Entrainement (GPU). Augmenter EPOCHS pour un vrai entrainement.
from src import config
config.EPOCHS = 25
from src.model import train
train.main()

In [ ]:
# 5. Evaluation sur le jeu de test + baseline Otsu
from src.model import evaluate
evaluate.main()

## (Bloc 6) Recherche d'hyperparametres
Compare plusieurs configurations (lr, filtres, taille) et sauvegarde la meilleure.

In [ ]:
from src.model import tune
tune.main()   # log MLflow + reports/hyperparam_search.json

## (Bloc 3.3) Analyse d'erreurs

In [ ]:
from src.model import error_analysis
error_analysis.main()

In [ ]:
# 6. Recuperer le modele entraine et les rapports
from google.colab import files
files.download('models/unet_caries.pth')
files.download('reports/hyperparam_search.json')